# 03 — Model Training
Train and evaluate all three models. Show learning curves, confusion matrices, and feature importance.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.metrics import confusion_matrix, classification_report, mean_absolute_error
from sklearn.model_selection import train_test_split

from train import load_and_prep, PlacementMLP, MODELS_DIR, PROCESSED_DIR

sns.set_theme(style='darkgrid')
DEVICE = torch.device('cpu')

In [ ]:
# Run training (or load saved model if already trained)
from train import train_mlp, train_clustering
features_path = PROCESSED_DIR / 'match_features.csv'
train_mlp(features_path, epochs=100)
train_clustering(features_path)

In [ ]:
# Evaluate MLP on held-out test set
with open(MODELS_DIR / 'mlp_encoders.pkl', 'rb') as f:
    encoders = pickle.load(f)

X, y, _ = load_and_prep(features_path)
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = PlacementMLP(input_dim=encoders['input_dim'])
model.load_state_dict(torch.load(MODELS_DIR / 'mlp_best.pt', map_location='cpu'))
model.eval()

with torch.no_grad():
    preds = model(torch.tensor(X_test)).argmax(dim=1).numpy()

print('Classification Report (8-class placement):')
print(classification_report(y_test, preds, target_names=[f'P{i+1}' for i in range(8)]))
print(f'MAE: {mean_absolute_error(y_test, preds):.3f}')
print(f'Top-4 binary accuracy: {((preds < 4) == (y_test < 4)).mean():.3f}')

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'P{i+1}' for i in range(8)],
            yticklabels=[f'P{i+1}' for i in range(8)], ax=ax)
ax.set_title('MLP Placement Confusion Matrix (Test Set)')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance via permutation (proxy: weight magnitude in first layer)
weights = model.net[0].weight.detach().abs().mean(dim=0).numpy()
feature_names = encoders['numeric_cols'] + encoders['categorical_cols']
importance = pd.Series(weights, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
importance.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Input Feature Importance (MLP first-layer weight magnitude)')
ax.set_ylabel('Mean |weight|')
plt.tight_layout()
plt.show()